In [1]:
# Run once
!pip install pandas tqdm

In [2]:
import pandas as pd
import email
from email import policy
from email.parser import BytesParser, Parser
import re
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ── change this to your actual path ──────────────────────────────
DATASET_PATH = "emails.csv"        # path to your downloaded Enron CSV
OUTPUT_PATH  = "email_reply_pairs.csv"
MAX_EMAILS   = None                # set e.g. 100_000 to test on subset first
# ───────────────────────────────────────────────────────────────────────────

print("Config loaded ✅")

Config loaded ✅


In [3]:
print("Loading dataset... (this may take ~30s for the full file)")

df_raw = pd.read_csv(DATASET_PATH, nrows=MAX_EMAILS)

print(f"Total emails loaded : {len(df_raw):,}")
print(f"Columns             : {list(df_raw.columns)}")
print("\nSample row (message field preview):")
print(df_raw['message'].iloc[0][:600])

Loading dataset... (this may take ~30s for the full file)
Total emails loaded : 517,401
Columns             : ['file', 'message']

Sample row (message field preview):
Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>
Date: Mon, 14 May 2001 16:39:00 -0700 (PDT)
From: phillip.allen@enron.com
To: tim.belden@enron.com
Subject: 
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: Tim Belden <Tim Belden/Enron@EnronXGate>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
X-Origin: Allen-P
X-FileName: pallen (Non-Privileged).pst

Here is our forecast

 


In [4]:
def parse_email(raw_message: str) -> dict:
    """
    Parse a raw RFC-2822 email string into a structured dict.
    Returns: message_id, in_reply_to, from_, to, subject, date, body
    """
    try:
        msg = Parser().parsestr(raw_message)
    except Exception:
        return None

    # ── Headers ──────────────────────────────────────────────────
    message_id   = msg.get("Message-ID", "").strip()
    in_reply_to  = msg.get("In-Reply-To", "").strip()
    from_        = msg.get("From", "").strip()
    to_          = msg.get("To", "").strip()
    subject      = msg.get("Subject", "").strip()
    date         = msg.get("Date", "").strip()

    # ── Body extraction ──────────────────────────────────────────
    body = ""
    if msg.is_multipart():
        for part in msg.walk():
            ctype = part.get_content_type()
            disp  = str(part.get("Content-Disposition", ""))
            if ctype == "text/plain" and "attachment" not in disp:
                try:
                    body = part.get_payload(decode=True).decode("utf-8", errors="replace")
                    break
                except Exception:
                    body = str(part.get_payload())
                    break
    else:
        try:
            payload = msg.get_payload(decode=True)
            body = payload.decode("utf-8", errors="replace") if payload else str(msg.get_payload())
        except Exception:
            body = str(msg.get_payload())

    return {
        "message_id"  : message_id,
        "in_reply_to" : in_reply_to,
        "from_"       : from_,
        "to_"         : to_,
        "subject"     : subject,
        "date"        : date,
        "body_raw"    : body
    }

print("Parser function defined ✅")

Parser function defined ✅


In [5]:
# ── Patterns to strip from email bodies ──────────────────────────────────
_QUOTE_LINE       = re.compile(r"^[ \t]*>.*$",          re.MULTILINE)
_ORIGINAL_MSG     = re.compile(r"-{3,}.*?Original Message.*",re.DOTALL | re.IGNORECASE)
_FORWARDED        = re.compile(r"-{3,}.*?Forwarded.*",  re.DOTALL | re.IGNORECASE)
_REPLIED          = re.compile(r"On .{10,80} wrote:",   re.DOTALL)
_SIGNATURE        = re.compile(r"\n--[ \t]*\n.*",       re.DOTALL)
_DISCLAIMER       = re.compile(
    r"(this (email|message) (is |may be )?(confidential|privileged).*)", 
    re.IGNORECASE | re.DOTALL
)
_MULTI_NEWLINE    = re.compile(r"\n{3,}")
_MULTI_SPACE      = re.compile(r"[ \t]{2,}")

def clean_body(text: str) -> str:
    """
    Strip quoted text, forwarded blocks, signatures, disclaimers.
    Returns clean, normalised plain text.
    """
    if not isinstance(text, str):
        return ""

    text = _QUOTE_LINE.sub("", text)          # remove >-quoted lines
    text = _ORIGINAL_MSG.sub("", text)         # remove -----Original...
    text = _FORWARDED.sub("", text)            # remove forwarded blocks
    text = _REPLIED.sub("", text)              # remove "On X wrote:"
    text = _SIGNATURE.sub("", text)            # remove -- signatures
    text = _DISCLAIMER.sub("", text)           # remove legal disclaimers

    # normalise whitespace
    text = _MULTI_NEWLINE.sub("\n\n", text)
    text = _MULTI_SPACE.sub(" ", text)
    text = text.strip()

    return text


def word_count(text: str) -> int:
    return len(text.split())


print("Cleaner function defined ✅")

Cleaner function defined ✅


In [6]:
print("Parsing all emails... grab a ☕")

parsed_emails = []
for raw in tqdm(df_raw['message'], desc="Parsing"):
    result = parse_email(raw)
    if result:
        parsed_emails.append(result)

df_parsed = pd.DataFrame(parsed_emails)

print(f"\nSuccessfully parsed : {len(df_parsed):,} emails")
print(f"Emails with replies : {(df_parsed['in_reply_to'] != '').sum():,}")
print(f"\nSample parsed email:")
print(df_parsed.iloc[100][['message_id','in_reply_to','subject','body_raw']].to_string())

Parsing all emails... grab a ☕


Parsing: 100%|██████████| 517401/517401 [00:46<00:00, 11235.10it/s]



Successfully parsed : 517,401 emails
Emails with replies : 0

Sample parsed email:
message_id         <15940494.1075855689309.JavaMail.evans@thyme>
in_reply_to                                                     
subject                                                      Re:
body_raw       I tried the new address but I don't have acces...


In [10]:
from datetime import datetime
import dateutil.parser

def normalize_subject(subject: str) -> str:
    """Strip Re:/Fwd: prefixes and normalize for grouping."""
    if not isinstance(subject, str):
        return ""
    s = subject.strip()
    # remove all Re: Fwd: Re[2]: etc.
    s = re.sub(r'^(re|fwd|fw|aw|sv|tr)[\[\(]?\d*[\]\)]?\s*:\s*', '', s,
               flags=re.IGNORECASE)
    s = s.strip().lower()
    return s

def parse_date(date_str: str):
    """Parse email date string to datetime, return None on failure."""
    if not isinstance(date_str, str) or not date_str.strip():
        return None
    try:
        return dateutil.parser.parse(date_str, fuzzy=True)
    except Exception:
        return None

# Apply to parsed df
df_parsed['subject_norm'] = df_parsed['subject'].apply(normalize_subject)
df_parsed['date_parsed']  = df_parsed['date'].apply(parse_date)

# Quick check
print(f"Subjects normalized ✅")
print(f"Dates parsed        : {df_parsed['date_parsed'].notna().sum():,} / {len(df_parsed):,}")
print(f"\nSample normalized subjects:")
print(df_parsed[['subject','subject_norm']].drop_duplicates().head(8).to_string(index=False))

Subjects normalized ✅
Dates parsed        : 517,401 / 517,401

Sample normalized subjects:
                                                    subject                                            subject_norm
                                                                                                                   
                                                        Re:                                                        
                                                   Re: test                                                    test
                                                  Re: Hello                                                   hello
                               Re: PRC review - phone calls                                prc review - phone calls
                             Re: High Speed Internet Access                              high speed internet access
    FW: fixed forward or other Collar floor gas price terms     fixed forward or other collar flo

In [11]:
print("Building pairs via subject-thread grouping...")
print("(may take 2–3 min on full dataset)\n")

pairs = []
skipped_no_date   = 0
skipped_too_short = 0
skipped_too_long  = 0
skipped_same_sender = 0

# Only work with emails that have a valid subject & date
df_valid = df_parsed[
    (df_parsed['subject_norm'].str.len() > 2) &
    (df_parsed['date_parsed'].notna())
].copy()

print(f"Emails with valid subject+date : {len(df_valid):,}")

# Group by normalized subject
groups = df_valid.groupby('subject_norm')

print(f"Unique subject threads         : {len(groups):,}")
print("Pairing within threads...\n")

for subj_norm, group in tqdm(groups, desc="Threading"):
    # need at least 2 emails in thread
    if len(group) < 2:
        continue

    # sort by date within thread
    group_sorted = group.sort_values('date_parsed').reset_index(drop=True)

    # pair each email with the next one in the thread
    for i in range(len(group_sorted) - 1):
        email_row = group_sorted.iloc[i]
        reply_row = group_sorted.iloc[i + 1]

        # sender should differ (not emailing yourself)
        if email_row['from_'].strip().lower() == reply_row['from_'].strip().lower():
            skipped_same_sender += 1
            continue

        # clean bodies
        email_body = clean_body(email_row['body_raw'])
        reply_body = clean_body(reply_row['body_raw'])

        email_wc = word_count(email_body)
        reply_wc = word_count(reply_body)

        # quality filters
        if email_wc < 5 or reply_wc < 3:
            skipped_too_short += 1
            continue
        if email_wc > 600 or reply_wc > 600:
            skipped_too_long += 1
            continue

        pairs.append({
            "email_id"    : email_row['message_id'],
            "reply_id"    : reply_row['message_id'],
            "email_from"  : email_row['from_'],
            "reply_from"  : reply_row['from_'],
            "subject"     : email_row['subject'],
            "subject_norm": subj_norm,
            "email_body"  : email_body,
            "reply_body"  : reply_body,
            "email_wc"    : email_wc,
            "reply_wc"    : reply_wc,
        })

df_pairs = pd.DataFrame(pairs)

print(f"✅ Pairs extracted       : {len(df_pairs):,}")
print(f"   Skipped (same sender) : {skipped_same_sender:,}")
print(f"   Skipped (too short)   : {skipped_too_short:,}")
print(f"   Skipped (too long)    : {skipped_too_long:,}")

if len(df_pairs) > 0:
    print(f"\nAvg email word count : {df_pairs['email_wc'].mean():.1f}")
    print(f"Avg reply word count : {df_pairs['reply_wc'].mean():.1f}")
else:
    print("\n⚠️  Still zero pairs — run the debug below")
    print("Subject norm samples:")
    print(df_valid['subject_norm'].value_counts().head(10))

Building pairs via subject-thread grouping...
(may take 2–3 min on full dataset)

Emails with valid subject+date : 482,543
Unique subject threads         : 128,046
Pairing within threads...



Threading: 100%|██████████| 128046/128046 [30:13<00:00, 70.59it/s]  


✅ Pairs extracted       : 48,184
   Skipped (same sender) : 289,946
   Skipped (too short)   : 14,628
   Skipped (too long)    : 1,739

Avg email word count : 89.8
Avg reply word count : 82.3


In [12]:
print("=" * 55)
print("         AUTOMAIL — PART 1 SUMMARY")
print("=" * 55)
print(f"  Raw emails loaded          : {len(df_raw):>10,}")
print(f"  Successfully parsed        : {len(df_parsed):>10,}")
print(f"  Valid email-reply pairs    : {len(df_pairs):>10,}")
print(f"  Unique senders (email)     : {df_pairs['email_from'].nunique():>10,}")
print(f"  Unique senders (reply)     : {df_pairs['reply_from'].nunique():>10,}")
print("=" * 55)

print("\n--- 3 sample pairs ---\n")
for i in range(3):
    row = df_pairs.iloc[i]
    print(f"[Subject]: {row['subject']}")
    print(f"[Email]  : {row['email_body'][:200]}")
    print(f"[Reply]  : {row['reply_body'][:150]}")
    print("-" * 50)

         AUTOMAIL — PART 1 SUMMARY
  Raw emails loaded          :    517,401
  Successfully parsed        :    517,401
  Valid email-reply pairs    :     48,184
  Unique senders (email)     :      8,943
  Unique senders (reply)     :      8,534

--- 3 sample pairs ---

[Subject]: !!!   OATI Etag 1.7 Minimum Requirements   !!!
[Email]  : Time is running very short. Is your company prepared for Etag 1.7? These are the minimum REQUIRED steps that must be completed prior to March 6, 2002:

---------------------------

(1) All entities mu
[Reply]  : Cara,

Do you know about this and are we ready for this version? 

Caroline
--------------------------------------------------
[Subject]: FW: !!!   OATI Etag 1.7 Minimum Requirements   !!!
[Email]  : Cara,

Do you know about this and are we ready for this version? 

Caroline
[Reply]  : Time is running very short. Is your company prepared for Etag 1.7? These are the minimum REQUIRED steps that must be completed prior to March 6, 2002:
-----------

In [13]:
df_pairs.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Saved {len(df_pairs):,} pairs → '{OUTPUT_PATH}'")
print(f"   File size: {os.path.getsize(OUTPUT_PATH) / 1024 / 1024:.1f} MB")

✅ Saved 48,184 pairs → 'email_reply_pairs.csv'
   File size: 58.6 MB
